# Model comparison - Letter, Digits và Covertype

Tổng hợp Decision Tree, Random Forest, SVM và KNN trên cả ba dataset. Trong từng
dataset, bốn mô hình dùng cùng train/test split. Notebook chỉ đọc result JSON;
không huấn luyện lại model. Trên Kaggle, hãy **Add Input** chứa sáu output ZIP
hoặc các JSON đã giải nén.


In [ ]:
import json
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None

    def display(value):
        print(value)

sns.set_theme(style="whitegrid")


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
KAGGLE_WORKING = Path("/kaggle/working")
RUN_ROOT = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
FIGURES_DIR = RUN_ROOT / "figures"
RESULTS_DIR = RUN_ROOT / "results"
for directory in (FIGURES_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID = "three_dataset_model_comparison"
EXPECTED_RESULTS = {
    "dt_letter_baseline.json",
    "dt_digits_baseline.json",
    "rf_letter_digits_benchmark.json",
    "svm_letter_digits_benchmark.json",
    "knn_letter_digits_benchmark.json",
    "covertype_four_model_benchmark.json",
}


def search_roots():
    roots = [Path("/kaggle/input"), Path.cwd() / "results", Path.cwd()]
    return [root for root in roots if root.exists()]


def read_result_json(filename):
    for root in search_roots():
        direct_matches = sorted(root.rglob(filename))
        if direct_matches:
            path = direct_matches[0]
            print(f"Loaded {filename}: {path}")
            return json.loads(path.read_text(encoding="utf-8"))
    for root in search_roots():
        for archive_path in sorted(root.rglob("*.zip")):
            try:
                with ZipFile(archive_path) as archive:
                    member = next(
                        (name for name in archive.namelist() if name.endswith(filename)), None
                    )
                    if member:
                        print(f"Loaded {filename}: {archive_path}!{member}")
                        return json.loads(archive.read(member).decode("utf-8"))
            except (OSError, ValueError):
                continue
    raise FileNotFoundError(
        f"Missing {filename}. Add Input chứa sáu output ZIP/JSON trước khi Run All."
    )


loaded = {name: read_result_json(name) for name in EXPECTED_RESULTS}

covertype_required_models = {"Decision Tree", "Random Forest", "KNN", "SVM (RBF)"}
covertype_result = loaded["covertype_four_model_benchmark.json"]
covertype_completed_models = set(covertype_result.get("models", {}))
covertype_missing_models = covertype_required_models - covertype_completed_models
if covertype_missing_models:
    raise ValueError(
        "Covertype benchmark is incomplete. Missing completed models: "
        + ", ".join(sorted(covertype_missing_models))
        + ". Re-run notebook 07 and use its final output ZIP."
    )


In [ ]:
records = []
for filename, result in loaded.items():
    if filename.startswith("dt_"):
        metrics = result["metrics"]
        records.append(
            {
                "dataset": result["dataset"],
                "model": "Decision Tree",
                "train_accuracy": metrics["train_accuracy"],
                "test_accuracy": metrics["test_accuracy"],
                "f1_macro": metrics["f1_macro"],
                "error_rate": metrics["error_rate"],
                "generalization_gap": metrics["train_accuracy"] - metrics["test_accuracy"],
                "training_seconds": metrics["training_seconds"],
                "prediction_seconds": metrics["prediction_seconds"],
                "total_model_seconds": (
                    metrics["training_seconds"] + metrics["prediction_seconds"]
                ),
                "dataset_scope": "full",
                "train_accuracy_scope": "entire_train_set",
                "learning_strategy": "eager",
            }
        )
    elif filename == "covertype_four_model_benchmark.json":
        for model_name, metrics in result["models"].items():
            records.append(
                {
                    "dataset": "covertype",
                    "model": model_name,
                    "train_accuracy": metrics["train_accuracy"],
                    "test_accuracy": metrics["test_accuracy"],
                    "f1_macro": metrics["f1_macro"],
                    "error_rate": metrics["error_rate"],
                    "generalization_gap": metrics["generalization_gap"],
                    "training_seconds": metrics["training_seconds"],
                    "prediction_seconds": metrics["prediction_seconds"],
                    "total_model_seconds": metrics["total_model_seconds"],
                    "dataset_scope": result["dataset_scope"],
                    "train_accuracy_scope": result["train_accuracy_scope"]["kind"],
                    "learning_strategy": metrics["learning_strategy"],
                }
            )
    else:
        for dataset_name, metrics in result["datasets"].items():
            records.append(
                {
                    "dataset": dataset_name,
                    "model": result["model"],
                    "train_accuracy": metrics["train_accuracy"],
                    "test_accuracy": metrics["test_accuracy"],
                    "f1_macro": metrics["f1_macro"],
                    "error_rate": metrics["error_rate"],
                    "generalization_gap": metrics["generalization_gap"],
                    "training_seconds": metrics["training_seconds"],
                    "prediction_seconds": metrics["prediction_seconds"],
                    "total_model_seconds": (
                        metrics["training_seconds"] + metrics["prediction_seconds"]
                    ),
                    "dataset_scope": "full",
                    "train_accuracy_scope": "entire_train_set",
                    "learning_strategy": "lazy" if result["model"] == "KNN" else "eager",
                }
            )

comparison = pd.DataFrame(records).sort_values(["dataset", "f1_macro"], ascending=[True, False])
display(comparison.round(4))

best_by_dataset = (
    comparison.loc[comparison.groupby("dataset")["f1_macro"].idxmax()]
    .set_index("dataset")[["model", "test_accuracy", "f1_macro"]]
    .to_dict(orient="index")
)
print("Best by macro-F1:", best_by_dataset)

insights = {}
for dataset_name, frame in comparison.groupby("dataset"):
    best = frame.loc[frame["f1_macro"].idxmax()]
    decision_tree = frame.loc[frame["model"] == "Decision Tree"].iloc[0]
    fastest_train = frame.loc[frame["training_seconds"].idxmin()]
    eager_frame = frame.loc[frame["learning_strategy"] == "eager"]
    fastest_eager_train = eager_frame.loc[eager_frame["training_seconds"].idxmin()]
    fastest_predict = frame.loc[frame["prediction_seconds"].idxmin()]
    fastest_total = frame.loc[frame["total_model_seconds"].idxmin()]
    smallest_gap = frame.loc[frame["generalization_gap"].idxmin()]
    insights[dataset_name] = {
        "best_macro_f1_model": best["model"],
        "best_macro_f1": float(best["f1_macro"]),
        "macro_f1_gain_over_decision_tree": float(
            best["f1_macro"] - decision_tree["f1_macro"]
        ),
        "fastest_training_model": fastest_train["model"],
        "fastest_eager_training_model": fastest_eager_train["model"],
        "fastest_prediction_model": fastest_predict["model"],
        "fastest_fit_plus_prediction_model": fastest_total["model"],
        "smallest_generalization_gap_model": smallest_gap["model"],
    }
display(pd.DataFrame(insights).T)


## Nguyên tắc đọc kết quả

Không chọn model chỉ theo accuracy. So sánh đồng thời macro-F1, train-test gap,
thời gian huấn luyện và suy luận. Với KNN cần ưu tiên tổng `fit + prediction` vì
đây là lazy learner. Kết quả test dùng để báo cáo, không dùng để tuning.


In [ ]:
artifact_paths = []

metrics_long = comparison.melt(
    id_vars=["dataset", "model"],
    value_vars=["test_accuracy", "f1_macro"],
    var_name="metric",
    value_name="score",
)
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
for ax, (dataset_name, frame) in zip(axes, metrics_long.groupby("dataset"), strict=True):
    sns.barplot(data=frame, x="model", y="score", hue="metric", ax=ax)
    ax.set(title=dataset_name.replace("_", " ").title(), ylim=(0.0, 1.0), xlabel="")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Model performance on the shared test splits")
fig.tight_layout()
performance_path = FIGURES_DIR / f"{EXPERIMENT_ID}__performance.png"
fig.savefig(performance_path, dpi=200, bbox_inches="tight")
plt.show()
artifact_paths.append(performance_path)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=comparison, x="model", y="generalization_gap", hue="dataset", ax=axes[0])
axes[0].set(title="Generalization gap", xlabel="", ylabel="Train accuracy - test accuracy")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(
    data=comparison,
    x="dataset",
    y="total_model_seconds",
    hue="model",
    errorbar=None,
    ax=axes[1],
)
axes[1].set(
    title="Fit + full-test prediction (log scale)",
    xlabel="",
    ylabel="Seconds",
    yscale="log",
)
axes[1].tick_params(axis="x", rotation=15)
fig.tight_layout()
tradeoff_path = FIGURES_DIR / f"{EXPERIMENT_ID}__gap_and_runtime.png"
fig.savefig(tradeoff_path, dpi=200, bbox_inches="tight")
plt.show()
artifact_paths.append(tradeoff_path)


In [ ]:
comparison_path = RESULTS_DIR / f"{EXPERIMENT_ID}.csv"
comparison.to_csv(comparison_path, index=False)
summary_path = RESULTS_DIR / f"{EXPERIMENT_ID}.json"
summary = {
    "schema_version": "1.0",
    "experiment_id": EXPERIMENT_ID,
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": True},
    "selection_metric": "macro-F1 on each dataset's shared test split (reporting only)",
    "timing_note": (
        "KNN is a lazy learner; use fit + full-test prediction for end-to-end comparison. "
        "Covertype generalization gap uses a fixed 10k stratified train diagnostic sample."
    ),
    "best_by_dataset": best_by_dataset,
    "insights": insights,
    "records": json.loads(comparison.to_json(orient="records")),
    "created_at_utc": datetime.now(UTC).isoformat(),
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
artifact_paths.extend([comparison_path, summary_path])

archive_path = RUN_ROOT / f"{EXPERIMENT_ID}__outputs.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for artifact_path in artifact_paths:
        archive.write(artifact_path, artifact_path.relative_to(RUN_ROOT))

print(f"Created ZIP: {archive_path}")
if FileLink is not None:
    display(FileLink(str(archive_path)))
